# Run the Document VQA project on Colab
Enable a GPU. Upload Document_VQA_Project.zip to /content/. This notebook does not retrain the model.


**Translation note:** This notebook is an English translation of the supplied notebook. Saved output messages were translated for readability; numerical results were preserved. Translation does not represent a new training or evaluation run.


In [ ]:
from pathlib import Path
import zipfile
from google.colab import drive
drive.mount('/content/drive')
zip_path = Path('/content/Document_VQA_Project.zip')
assert zip_path.exists(), 'Upload the project ZIP to /content first.'
with zipfile.ZipFile(zip_path) as z:
    z.extractall('/content/vqa_code')
CODE_DIR = Path('/content/vqa_code/Document_VQA_Project')
print(CODE_DIR)


Mounted at /content/drive
/content/vqa_code/Document_VQA_Project


In [ ]:
%pip install -q -r /content/vqa_code/Document_VQA_Project/requirements.txt
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-eng


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 111.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 40.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 16.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.

If a restart is required, or libraries were imported before installation, select Runtime → Restart session, then run the cell below. Do not rerun training. This demo does not use Gradio or Diffusers.


In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
CODE_DIR = Path('/content/vqa_code/Document_VQA_Project')
PROJECT_DATA_DIR = Path('/content/drive/MyDrive/Document_VQA_Project')
MODEL_DIR = PROJECT_DATA_DIR / 'checkpoints/donut_decoder_finetuned'
assert (MODEL_DIR / 'config.json').exists(), 'Check the checkpoint path on Drive.'
assert list(MODEL_DIR.glob('*.safetensors')) or (MODEL_DIR / 'pytorch_model.bin').exists(), 'Missing model weights.'
print('Checkpoint found:', MODEL_DIR)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoint found: /content/drive/MyDrive/Document_VQA_Project/checkpoints/donut_decoder_finetuned


In [ ]:
import subprocess, sys
# Run the demo in a separate process to avoid old libraries in the kernel.
# Stop the old server before rerunning this cell.
if 'demo_process' in globals() and demo_process.poll() is None:
    demo_process.terminate()
    demo_process.wait(timeout=15)
demo_log = open('/content/vqa_demo.log', 'w')
demo_process = subprocess.Popen(
    [sys.executable, 'app.py', '--model-path', str(MODEL_DIR),
     '--host', '0.0.0.0', '--port', '5000'],
    cwd=CODE_DIR, stdout=demo_log, stderr=subprocess.STDOUT,
)
print('Demo started. Log: /content/vqa_demo.log')


Demo started. Log: /content/vqa_demo.log


In [ ]:
import time, urllib.request
from google.colab import output
ready = False
for attempt in range(30):
    if demo_process.poll() is not None:
        break
    try:
        with urllib.request.urlopen('http://127.0.0.1:5000/', timeout=2) as r:
            ready = r.status == 200
        if ready:
            break
    except OSError:
        time.sleep(1)
if ready:
    output.serve_kernel_port_as_iframe(5000, height=850)
else:
    print(Path('/content/vqa_demo.log').read_text()[-5000:])


<IPython.core.display.Javascript object>

Select an image from Drive data/images or receipts/images and ask a question with a known answer for an actual check. Demo timing excludes initial model loading.


In [ ]:
# Optional: rescore saved raw JSON files without running the models.
import subprocess, sys
subprocess.run(
    [sys.executable, '-m', 'src.summarize_results',
     '--results-dir', str(PROJECT_DATA_DIR / 'results')],
    cwd=CODE_DIR, check=True,
)


In [ ]:
from pathlib import Path
import shutil
import zipfile

PROJECT_DIR = Path("/content/drive/MyDrive/Document_VQA_Project")
EXPORT_DIR = PROJECT_DIR / "submission"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

sources = {
    "processed_dataset": PROJECT_DIR / "data",
    "donut_decoder_finetuned": (
        PROJECT_DIR / "checkpoints" / "donut_decoder_finetuned"
    ),
}

for name, source in sources.items():
    assert source.is_dir(), f"Not found: {source}"

    destination = EXPORT_DIR / f"{name}.zip"

    if destination.exists():
        print("Already exists; preserving:", destination)
        continue

    # Create the ZIP on Colab storage first, then copy it to Drive
    local_zip = shutil.make_archive(
        str(Path("/content") / name),
        "zip",
        root_dir=source.parent,
        base_dir=source.name,
    )

    with zipfile.ZipFile(local_zip) as archive:
        assert archive.testzip() is None, f"Invalid ZIP archive: {name}"

    shutil.copy2(local_zip, destination)
    print("Saved:", destination)
    print(f"Size: {destination.stat().st_size / 1024**2:.1f} MB")

Saved: /content/drive/MyDrive/Document_VQA_Project/submission/processed_dataset.zip
Size: 224.6 MB
Saved: /content/drive/MyDrive/Document_VQA_Project/submission/donut_decoder_finetuned.zip
Size: 710.8 MB


In [4]:
from pathlib import Path

matches = [
    p for p in Path("/content").rglob("train_donut.py")
    if p.parent.name == "src"
]

if matches:
    for p in matches:
        print("Source found:", p.parent.parent)
else:
    print("train_donut.py was not found in this Colab session.")
    print("Available archive files:")
    for pattern in ("*.zip", "*.rar"):
        for p in Path("/content").glob(pattern):
            print(p)

Source found: /content/vqa_code/Document_VQA_Project


In [3]:
from pathlib import Path
import zipfile

ZIP = Path("/content/Document_VQA_Project.zip")
DEST = Path("/content/vqa_code")

assert ZIP.is_file(), "Upload Document_VQA_Project.zip first"

with zipfile.ZipFile(ZIP) as archive:
    archive.extractall(DEST)

matches = [
    p for p in DEST.rglob("train_donut.py")
    if p.parent.name == "src"
]

assert len(matches) == 1, (
    f"Found {len(matches)} train_donut.py files; check the ZIP structure"
)

CODE = matches[0].parent.parent
print("Source:", CODE)
print("Training script exists:", (CODE / "src/train_donut.py").is_file())

Source: /content/vqa_code/Document_VQA_Project
Training script exists: True


In [5]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import subprocess
import torch

PROJECT = Path("/content/drive/MyDrive/Document_VQA_Project")
CODE = Path("/content/vqa_code/Document_VQA_Project")
BASE_MODEL = Path("/content/local_models/DonutDocVQA")
RAR = PROJECT / "checkpoints/DonutDocVQA.rar"

assert torch.cuda.is_available(), "Enable a GPU in Colab"
assert (CODE / "src/train_donut.py").is_file(), "Source code is missing"
assert (PROJECT / "data_clean/train.json").is_file(), "Missing data_clean"

if not (BASE_MODEL / "config.json").is_file():
    assert RAR.is_file(), f"Initial model not found: {RAR}"

    subprocess.run(["apt-get", "install", "-y", "unrar"], check=True)
    BASE_MODEL.parent.mkdir(parents=True, exist_ok=True)

    subprocess.run(
        ["unrar", "x", "-o+", str(RAR), str(BASE_MODEL.parent) + "/"],
        check=True,
    )

assert (BASE_MODEL / "config.json").is_file(), (
    "config.json was not found at the expected model path"
)
assert (
    (BASE_MODEL / "pytorch_model.bin").is_file()
    or (BASE_MODEL / "model.safetensors").is_file()
), "Missing initial model weights"

print("GPU:", torch.cuda.get_device_name(0))
print("Initial model:", BASE_MODEL)
print("Dataset:", PROJECT / "data_clean")

Mounted at /content/drive
GPU: Tesla T4
Initial model: /content/local_models/DonutDocVQA
Dataset: /content/drive/MyDrive/Document_VQA_Project/data_clean


In [6]:
import os
import sys
import subprocess

env = os.environ.copy()
env["HF_HUB_DISABLE_XET"] = "1"

subprocess.run(
    [
        sys.executable, "-u", "-m", "src.train_donut",
        "--data-dir", str(PROJECT / "data_clean"),
        "--model-path", str(BASE_MODEL),
        "--output-dir", str(
            PROJECT / "checkpoints/donut_decoder_finetuned_clean"
        ),
        "--results-dir", str(PROJECT / "results/clean_run"),
        "--epochs", "2",
        "--learning-rate", "1e-5",
    ],
    cwd=str(CODE),
    env=env,
    check=True,
)


CompletedProcess(args=['/usr/bin/python3', '-u', '-m', 'src.train_donut', '--data-dir', '/content/drive/MyDrive/Document_VQA_Project/data_clean', '--model-path', '/content/local_models/DonutDocVQA', '--output-dir', '/content/drive/MyDrive/Document_VQA_Project/checkpoints/donut_decoder_finetuned_clean', '--results-dir', '/content/drive/MyDrive/Document_VQA_Project/results/clean_run', '--epochs', '2', '--learning-rate', '1e-5'], returncode=0)

In [7]:
from pathlib import Path
import json

PROJECT = Path("/content/drive/MyDrive/Document_VQA_Project")
CHECKPOINT = PROJECT / "checkpoints/donut_decoder_finetuned_clean"
HISTORY = PROJECT / "results/clean_run/training_history.json"

assert HISTORY.is_file(), "Training history not found"
assert (CHECKPOINT / "config.json").is_file(), "Missing model configuration"
assert (CHECKPOINT / "model.safetensors").is_file(), "Missing model weights"

history = json.loads(HISTORY.read_text())
for row in history:
    print(row)

print("\nCheckpoint:", CHECKPOINT)
print("✅ Configuration and weights are present")

{'epoch': 0, 'validation_loss': 5.6370471739307435}
{'epoch': 1, 'train_loss': 3.9302242286520106, 'validation_loss': 2.942352290824544}
{'epoch': 2, 'train_loss': 2.2003518663659407, 'validation_loss': 2.2973048657722694}

Checkpoint: /content/drive/MyDrive/Document_VQA_Project/checkpoints/donut_decoder_finetuned_clean
✅ Configuration and weights are present


In [8]:
import os
import sys
import subprocess
import json
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/Document_VQA_Project")
CODE = Path("/content/vqa_code/Document_VQA_Project")
CHECKPOINT = PROJECT / "checkpoints/donut_decoder_finetuned_clean"
OUTPUT = PROJECT / "results/clean_run/donut_after_finetune.json"

assert not OUTPUT.exists(), "Results already exist; refusing to overwrite"

env = os.environ.copy()
env["HF_HUB_DISABLE_XET"] = "1"

command = [
    sys.executable, "-u", "-m", "src.evaluate",
    "--method", "donut",
    "--model-path", str(CHECKPOINT),
    "--data-file", str(PROJECT / "data_clean/evaluation.json"),
    "--output", str(OUTPUT),
    "--device", "cuda",
]

with subprocess.Popen(
    command,
    cwd=str(CODE),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
) as process:
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f"Evaluation failed with exit code {return_code}")

# Compute metrics using the project code
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

from src.metrics import summarize

rows = json.loads(OUTPUT.read_text())
metrics = summarize(rows)

print("\n--- NEW CHECKPOINT RESULTS ---")
print(f"Correct: {metrics['correct']}/{metrics['questions']}")
print(f"Exact Match: {metrics['em_percent']:.2f}%")
print(f"ANLS: {metrics['anls']:.4f}")
print(f"Time: {metrics['seconds_per_question']:.2f} seconds/question")
print("Saved:", OUTPUT)

[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.

Loading weights: 100%|██████████| 483/483 [00:00<00:00, 2721.05it/s]
1/57
2/57
3/57
4/57
5/57
6/57
7/57
8/57
9/57
10/57
11/57
12/57
13/57
14/57
15/57
16/57
17/57
18/57
19/57
20/57
21/57
22/57
23/57
24/57
25/57
26/57
27/57
28/57
29/57
30/57
31/57
32/57
33/57
34/57
35/57
36/57
37/57
38/57
39/57
40/57
41/57
42/57
43/57
44/57
45/57
46/57
47/57
48/57
49/57
50/57
51/57
52/57
53/57
54/57
55/57
56/57
57/57
{
  "questions": 57,
  "correct": 42,
  "em_percent": 73.6842105263158,
  "anls": 0.7947994987468672,
  "seconds_per_question": 1.490688242105283
}

--- NEW CHECKPOINT RESULTS ---
Correct: 42/57
Exact Match: 73.68%
ANLS: 0.7948
Time: 1.49 seconds/question
Saved: /content/drive/MyDrive/Document_VQA_Project/results/clean_run/donut_after_finetune.json


In [9]:
import transformers
import torch
import huggingface_hub

print("Transformers:", transformers.__version__)
print("PyTorch:", torch.__version__)
print("Hugging Face Hub:", huggingface_hub.__version__)

Transformers: 5.18.0
PyTorch: 2.11.0+cu130
Hugging Face Hub: 1.33.0


In [14]:
from pathlib import Path
import os
import sys
import subprocess
import json

PROJECT = Path("/content/drive/MyDrive/Document_VQA_Project")
CODE = Path("/content/vqa_code/Document_VQA_Project")
RESULTS = PROJECT / "results/clean_run"
DATA_FILE = PROJECT / "data_clean/evaluation.json"
BASE_MODEL = Path("/content/local_models/DonutDocVQA")

RESULTS.mkdir(parents=True, exist_ok=True)

# Record the actual versions in the new session
import transformers
import torch
import huggingface_hub

versions = {
    "transformers": transformers.__version__,
    "torch": torch.__version__,
    "huggingface_hub": huggingface_hub.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
(RESULTS / "environment.json").write_text(
    json.dumps(versions, indent=2), encoding="utf-8"
)

env = os.environ.copy()
env["HF_HUB_DISABLE_XET"] = "1"

def run_with_logs(command):
    with subprocess.Popen(
        command, cwd=str(CODE), env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        code = process.wait()
    if code != 0:
        raise RuntimeError(f"Command failed with exit code {code}")

# Prepare Tesseract for the baseline
run_with_logs(["apt-get", "install", "-y", "tesseract-ocr"])

experiments = [
    ("donut_before_finetune", [
        "--method", "donut",
        "--model-path", str(BASE_MODEL),
    ]),
    ("ocr_qa_baseline", [
        "--method", "ocr",
    ]),
]

for name, arguments in experiments:
    output = RESULTS / f"{name}.json"

    if output.exists():
        print(f"\nAlready exists {output.name}, skipping to preserve results.")
        continue

    print(f"\nRunning: {name}", flush=True)
    run_with_logs([
        sys.executable, "-u", "-m", "src.evaluate",
        *arguments,
        "--data-file", str(DATA_FILE),
        "--output", str(output),
        "--device", "cuda",
    ])

# Summarize the three methods, including the newly evaluated checkpoint
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

from src.metrics import summarize

print("\n--- COMPARISON IN THE CURRENT ENVIRONMENT ---")
for name in [
    "ocr_qa_baseline",
    "donut_before_finetune",
    "donut_after_finetune",
]:
    rows = json.loads((RESULTS / f"{name}.json").read_text())
    metrics = summarize(rows)
    print(
        f"{name}: "
        f"{metrics['correct']}/{metrics['questions']} correct | "
        f"EM {metrics['em_percent']:.2f}% | "
        f"ANLS {metrics['anls']:.4f} | "
        f"{metrics['seconds_per_question']:.2f} seconds/question"
    )


Reading package lists...
Building dependency tree...
Reading state information...
tesseract-ocr is already the newest version (5.3.4-1build5).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.

Already exists donut_before_finetune.json, skipping to preserve results.

Running: ocr_qa_baseline

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7037.66it/s]
1/57
2/57
3/57
4/57
5/57
6/57
7/57
8/57
9/57
10/57
11/57
12/57
13/57
14/57
15/57
16/57
17/57
18/57
19/57
20/57
21/57
22/57
23/57
24/57
25/57
26/57
27/57
28/57
29/57
30/57
31/57
32/57
33/57
34/57
35/57
36/57
37/57
38/57
39/57
40/57
41/57
42/57
43/57
44/57
45/57
46/57
47/57
48/57
49/57
50/57
51/57
52/57
53/57
54/57
55/57
56/57
57/57
{
  "questions": 57,
  "correct": 13,
  "em_percent": 22.80701754385965,
  "anls": 0.2627923976608187,
  "seconds_per_question": 3.151594487333305
}

--- COMPARISON IN THE CURRENT ENVIRONMENT ---
ocr_qa_baseline: 13/57 correct | EM 22.81% | ANLS 0.2628 | 3.15 seconds/question
donut_before_

In [13]:
from pathlib import Path
import shutil

CODE = Path("/content/vqa_code/Document_VQA_Project")
target = CODE / "src/ocr_qa.py"
backup = CODE / "src/ocr_qa_pipeline_backup.py"

if not backup.exists():
    shutil.copy2(target, backup)

target.write_text('''
import torch
import pytesseract
from transformers import AutoTokenizer, AutoModelForQuestionAnswering

class OCRQA:
    def __init__(self, model_id="deepset/roberta-base-squad2",
                 device="auto", tesseract_cmd=None):
        if tesseract_cmd:
            pytesseract.pytesseract.tesseract_cmd = tesseract_cmd

        selected = (
            "cuda" if torch.cuda.is_available() else "cpu"
        ) if device == "auto" else device

        self.device = torch.device(selected)
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id, use_fast=True
        )
        self.model = AutoModelForQuestionAnswering.from_pretrained(
            model_id, use_safetensors=True
        ).to(self.device).eval()

    def predict(self, image, question):
        context = pytesseract.image_to_string(
            image.convert("RGB"), lang="eng",
            config="--oem 3 --psm 3"
        ).strip()

        if not context:
            return {"answer": "", "qa_score": 0.0, "ocr_text": ""}

        # Limit question length and split long text into windows
        q_ids = self.tokenizer(
            question, add_special_tokens=False
        )["input_ids"]
        if len(q_ids) > 64:
            question = self.tokenizer.decode(
                q_ids[:64], skip_special_tokens=True
            )

        encoded = self.tokenizer(
            question, context,
            max_length=384,
            truncation="only_second",
            stride=128,
            return_overflowing_tokens=True,
            return_offsets_mapping=True,
            padding="max_length",
            return_tensors="pt",
        )

        offsets = encoded.pop("offset_mapping")
        encoded.pop("overflow_to_sample_mapping")
        best_score, best_answer = -1.0, ""

        with torch.inference_mode():
            for window in range(encoded["input_ids"].shape[0]):
                inputs = {
                    key: value[window:window + 1].to(self.device)
                    for key, value in encoded.items()
                }
                output = self.model(**inputs)

                sequence_ids = encoded.sequence_ids(window)
                context_mask = torch.tensor(
                    [value == 1 for value in sequence_ids],
                    device=self.device,
                    dtype=torch.bool,
                )
                valid = context_mask & inputs["attention_mask"][0].bool()

                # Allow the CLS token when normalizing probabilities,
                # but do not select an empty answer.
                cls_matches = (
                    inputs["input_ids"][0]
                    == self.tokenizer.cls_token_id
                ).nonzero(as_tuple=True)[0]
                allowed = valid.clone()
                if len(cls_matches):
                    allowed[cls_matches[0]] = True

                start = output.start_logits[0].masked_fill(
                    ~allowed, -10000.0
                ).softmax(dim=-1)
                end = output.end_logits[0].masked_fill(
                    ~allowed, -10000.0
                ).softmax(dim=-1)

                start = start.masked_fill(~valid, 0)
                end = end.masked_fill(~valid, 0)

                scores = start[:, None] * end[None, :]
                length = scores.shape[0]
                positions = torch.arange(
                    length, device=self.device
                )
                span_length = (
                    positions[None, :] - positions[:, None] + 1
                )
                scores = scores.masked_fill(
                    (span_length < 1) | (span_length > 30), 0
                )

                score, index = scores.flatten().max(dim=0)
                if score.item() > best_score and score.item() > 0:
                    first = index.item() // length
                    last = index.item() % length
                    begin = offsets[window, first, 0].item()
                    finish = offsets[window, last, 1].item()
                    best_answer = context[begin:finish]
                    best_score = score.item()

        return {
            "answer": best_answer.strip(),
            "qa_score": max(best_score, 0.0),
            "ocr_text": context,
        }
'''.lstrip(), encoding="utf-8")

print("Updated:", target)
print("Previous code:", backup)

Updated: /content/vqa_code/Document_VQA_Project/src/ocr_qa.py
Previous code: /content/vqa_code/Document_VQA_Project/src/ocr_qa_pipeline_backup.py


In [11]:
%pip install -q "pytesseract==0.3.13"

In [15]:
jobs = [
    ("ocr_qa_downsample50", "ocr",
     PROJECT / "data_clean/evaluation.json", "downsample50"),
    ("donut_after_finetune_downsample50", "donut",
     PROJECT / "data_clean/evaluation.json", "downsample50"),
    ("receipt_ocr_qa", "ocr",
     PROJECT / "data_clean/receipts/evaluation.json", "original"),
    ("receipt_donut_after_finetune", "donut",
     PROJECT / "data_clean/receipts/evaluation.json", "original"),
]

for name, method, data_file, transform in jobs:
    output = RESULTS / f"{name}.json"

    if output.exists():
        print(f"Already exists {name}, skipping.")
        continue

    command = [
        sys.executable, "-u", "-m", "src.evaluate",
        "--method", method,
        "--data-file", str(data_file),
        "--transform", transform,
        "--output", str(output),
        "--device", "cuda",
    ]

    if method == "donut":
        command += [
            "--model-path",
            str(PROJECT / "checkpoints/donut_decoder_finetuned_clean"),
        ]

    print(f"\nRunning: {name}", flush=True)
    run_with_logs(command)

print("\n--- ADDITIONAL RESULTS ---")
for name, _, _, _ in jobs:
    rows = json.loads((RESULTS / f"{name}.json").read_text())
    m = summarize(rows)
    print(
        f"{name}: {m['correct']}/{m['questions']} correct | "
        f"EM {m['em_percent']:.2f}% | ANLS {m['anls']:.4f} | "
        f"{m['seconds_per_question']:.2f} seconds/question"
    )


Running: ocr_qa_downsample50

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14598.71it/s]
1/57
2/57
3/57
4/57
5/57
6/57
7/57
8/57
9/57
10/57
11/57
12/57
13/57
14/57
15/57
16/57
17/57
18/57
19/57
20/57
21/57
22/57
23/57
24/57
25/57
26/57
27/57
28/57
29/57
30/57
31/57
32/57
33/57
34/57
35/57
36/57
37/57
38/57
39/57
40/57
41/57
42/57
43/57
44/57
45/57
46/57
47/57
48/57
49/57
50/57
51/57
52/57
53/57
54/57
55/57
56/57
57/57
{
  "questions": 57,
  "correct": 14,
  "em_percent": 24.56140350877193,
  "anls": 0.28688197767145135,
  "seconds_per_question": 3.0419165988420995
}

Running: donut_after_finetune_downsample50
[transformers] The `use_fast` parameter is deprecated and will be removed in a future version. Use `backend="torchvision"` instead of `use_fast=True`, or `backend="pil"` instead of `use_fast=False`.

Loading weights: 100%|██████████| 483/483 [00:00<00:00, 2910.61it/s]
1/57
2/57
3/57
4/57
5/57
6/57
7/57
8/57
9/57
10/57
11/57
12/57
13/57
14/57
15/57
16/57
17/57
18/57
19/

In [16]:
run_with_logs([
    sys.executable, "-u", "-m", "src.summarize_results",
    "--results-dir", str(RESULTS),
])

print("\nSummary table:", RESULTS / "all_metrics.csv")

{'experiment': 'ocr_qa_baseline', 'questions': 57, 'correct': 13, 'em_percent': 22.80701754385965, 'anls': 0.2627923976608187, 'seconds_per_question': 3.151594487333305}
{'experiment': 'donut_before_finetune', 'questions': 57, 'correct': 40, 'em_percent': 70.17543859649123, 'anls': 0.7738582567529936, 'seconds_per_question': 1.5254402969999799}
{'experiment': 'donut_after_finetune', 'questions': 57, 'correct': 42, 'em_percent': 73.6842105263158, 'anls': 0.7947994987468672, 'seconds_per_question': 1.490688242105283}
{'experiment': 'ocr_qa_downsample50', 'questions': 57, 'correct': 14, 'em_percent': 24.56140350877193, 'anls': 0.28688197767145135, 'seconds_per_question': 3.0419165988420995}
{'experiment': 'donut_after_finetune_downsample50', 'questions': 57, 'correct': 41, 'em_percent': 71.9298245614035, 'anls': 0.7833124477861321, 'seconds_per_question': 1.4510802903157782}
{'experiment': 'receipt_ocr_qa', 'questions': 40, 'correct': 2, 'em_percent': 5.0, 'anls': 0.09722222222222224, 'se

In [17]:
import shutil
from pathlib import Path
from google.colab import files

PROJECT = Path("/content/drive/MyDrive/Document_VQA_Project")
CODE = Path("/content/vqa_code/Document_VQA_Project")
EXPORT = Path("/content/clean_run_update")

EXPORT.mkdir(exist_ok=True)
(EXPORT / "src").mkdir(exist_ok=True)

shutil.copy2(CODE / "src/ocr_qa.py", EXPORT / "src/ocr_qa.py")
shutil.copytree(
    PROJECT / "results/clean_run",
    EXPORT / "results/clean_run",
    dirs_exist_ok=True,
)

archive = shutil.make_archive(
    "/content/clean_run_update", "zip", root_dir=EXPORT
)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
from pathlib import Path
import shutil
import zipfile

PROJECT = Path("/content/drive/MyDrive/Document_VQA_Project")
SUBMISSION = PROJECT / "submission"
SUBMISSION.mkdir(exist_ok=True)

sources = {
    "processed_dataset_clean": PROJECT / "data_clean",
    "donut_decoder_finetuned_clean":
        PROJECT / "checkpoints/donut_decoder_finetuned_clean",
}

for name, source in sources.items():
    assert source.is_dir(), f"Missing directory: {source}"
    destination = SUBMISSION / f"{name}.zip"

    if destination.exists():
        print("Already exists; not overwriting:", destination)
        continue

    local_zip = shutil.make_archive(
        str(Path("/content") / name),
        "zip",
        root_dir=source.parent,
        base_dir=source.name,
    )

    with zipfile.ZipFile(local_zip) as archive:
        assert archive.testzip() is None, "Invalid ZIP archive"

    shutil.copy2(local_zip, destination)
    print("Saved:", destination)

Saved: /content/drive/MyDrive/Document_VQA_Project/submission/processed_dataset_clean.zip
Saved: /content/drive/MyDrive/Document_VQA_Project/submission/donut_decoder_finetuned_clean.zip
